# N-Gram Language Models
## Unigram, Bigram, Trigram with Smoothing

This notebook demonstrates:
1. **N-gram models**: Unigram, Bigram, and Trigram
2. **Probability calculation** for word sequences
3. **Smoothing techniques**: Laplace (Add-1) and Add-k smoothing
4. **Perplexity evaluation** of language models

N-gram models predict the next word based on the previous n-1 words.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
from itertools import islice
import warnings
warnings.filterwarnings('ignore')

## Sample Corpus

We'll use a small corpus to demonstrate the concepts clearly.

In [ ]:
# Training corpus
train_corpus = [
    "I love natural language processing",
    "I love machine learning",
    "natural language processing is amazing",
    "machine learning is powerful",
    "I enjoy learning new things",
    "deep learning is a subset of machine learning",
    "natural language understanding is challenging"
]

# Test sentences
test_sentences = [
    "I love machine learning",
    "natural language processing is powerful"
]

print("Training Corpus:")
for i, sent in enumerate(train_corpus, 1):
    print(f"{i}. {sent}")

print("\nTest Sentences:")
for i, sent in enumerate(test_sentences, 1):
    print(f"{i}. {sent}")

In [ ]:
# Preprocessing function
def preprocess(text):
    """Convert to lowercase and split into tokens"""
    return text.lower().split()

# Tokenize corpus
train_tokens = [preprocess(sent) for sent in train_corpus]

print("Tokenized corpus:")
for tokens in train_tokens:
    print(tokens)

## 1. Unigram Model

The simplest n-gram model where each word is independent.

**Probability formula**: P(w) = Count(w) / Total words

**Sentence probability**: P(w₁, w₂, ..., wₙ) = P(w₁) × P(w₂) × ... × P(wₙ)

In [ ]:
class UnigramModel:
    def __init__(self):
        self.word_counts = Counter()
        self.total_words = 0
        self.vocab = set()
    
    def train(self, tokenized_corpus):
        """Train unigram model on tokenized corpus"""
        for tokens in tokenized_corpus:
            self.word_counts.update(tokens)
            self.total_words += len(tokens)
            self.vocab.update(tokens)
    
    def probability(self, word):
        """Calculate P(word)"""
        return self.word_counts[word] / self.total_words if self.total_words > 0 else 0
    
    def sentence_probability(self, tokens):
        """Calculate probability of a sentence"""
        prob = 1.0
        for word in tokens:
            word_prob = self.probability(word)
            if word_prob == 0:
                return 0  # Zero probability for unseen words
            prob *= word_prob
        return prob
    
    def log_probability(self, tokens):
        """Calculate log probability (more stable)"""
        log_prob = 0.0
        for word in tokens:
            word_prob = self.probability(word)
            if word_prob == 0:
                return float('-inf')
            log_prob += np.log(word_prob)
        return log_prob

In [ ]:
# Train unigram model
unigram = UnigramModel()
unigram.train(train_tokens)

print(f"Vocabulary size: {len(unigram.vocab)}")
print(f"Total words: {unigram.total_words}")
print("\nTop 10 most frequent words:")
for word, count in unigram.word_counts.most_common(10):
    prob = unigram.probability(word)
    print(f"{word:15s}: count={count:2d}, P(w)={prob:.4f}")

In [ ]:
# Test unigram model
test_tokens = preprocess(test_sentences[0])
print(f"Test sentence: '{test_sentences[0]}'")
print(f"Tokens: {test_tokens}")
print(f"\nProbability: {unigram.sentence_probability(test_tokens):.2e}")
print(f"Log probability: {unigram.log_probability(test_tokens):.4f}")

## 2. Bigram Model

Predicts each word based on the previous word.

**Probability formula**: P(wᵢ | wᵢ₋₁) = Count(wᵢ₋₁, wᵢ) / Count(wᵢ₋₁)

**Sentence probability**: P(w₁, w₂, ..., wₙ) = P(w₁) × P(w₂|w₁) × P(w₃|w₂) × ... × P(wₙ|wₙ₋₁)

In [ ]:
class BigramModel:
    def __init__(self):
        self.bigram_counts = defaultdict(Counter)
        self.unigram_counts = Counter()
        self.vocab = set()
    
    def train(self, tokenized_corpus):
        """Train bigram model on tokenized corpus"""
        for tokens in tokenized_corpus:
            # Add start token
            tokens = ['<START>'] + tokens
            
            # Count bigrams
            for i in range(len(tokens) - 1):
                self.bigram_counts[tokens[i]][tokens[i+1]] += 1
                self.unigram_counts[tokens[i]] += 1
                self.vocab.add(tokens[i+1])
            
            # Count last unigram
            self.unigram_counts[tokens[-1]] += 1
    
    def probability(self, word, previous_word):
        """Calculate P(word | previous_word)"""
        if self.unigram_counts[previous_word] == 0:
            return 0
        return self.bigram_counts[previous_word][word] / self.unigram_counts[previous_word]
    
    def sentence_probability(self, tokens):
        """Calculate probability of a sentence"""
        tokens = ['<START>'] + tokens
        prob = 1.0
        
        for i in range(1, len(tokens)):
            bigram_prob = self.probability(tokens[i], tokens[i-1])
            if bigram_prob == 0:
                return 0
            prob *= bigram_prob
        return prob
    
    def log_probability(self, tokens):
        """Calculate log probability"""
        tokens = ['<START>'] + tokens
        log_prob = 0.0
        
        for i in range(1, len(tokens)):
            bigram_prob = self.probability(tokens[i], tokens[i-1])
            if bigram_prob == 0:
                return float('-inf')
            log_prob += np.log(bigram_prob)
        return log_prob

In [ ]:
# Train bigram model
bigram = BigramModel()
bigram.train(train_tokens)

print("Sample bigram probabilities:")
print("\nP(word | 'I'):")
for word, count in bigram.bigram_counts['i'].most_common():
    prob = bigram.probability(word, 'i')
    print(f"  P('{word}' | 'I') = {prob:.4f}")

print("\nP(word | 'learning'):")
for word, count in bigram.bigram_counts['learning'].most_common():
    prob = bigram.probability(word, 'learning')
    print(f"  P('{word}' | 'learning') = {prob:.4f}")

In [ ]:
# Test bigram model
test_tokens = preprocess(test_sentences[0])
print(f"Test sentence: '{test_sentences[0]}'")
print(f"Tokens: {test_tokens}")
print(f"\nProbability: {bigram.sentence_probability(test_tokens):.2e}")
print(f"Log probability: {bigram.log_probability(test_tokens):.4f}")

## 3. Trigram Model

Predicts each word based on the previous two words.

**Probability formula**: P(wᵢ | wᵢ₋₂, wᵢ₋₁) = Count(wᵢ₋₂, wᵢ₋₁, wᵢ) / Count(wᵢ₋₂, wᵢ₋₁)

**Sentence probability**: P(w₁, ..., wₙ) = P(w₁) × P(w₂|w₁) × P(w₃|w₁,w₂) × ... × P(wₙ|wₙ₋₂,wₙ₋₁)

In [ ]:
class TrigramModel:
    def __init__(self):
        self.trigram_counts = defaultdict(lambda: defaultdict(Counter))
        self.bigram_counts = defaultdict(Counter)
        self.vocab = set()
    
    def train(self, tokenized_corpus):
        """Train trigram model on tokenized corpus"""
        for tokens in tokenized_corpus:
            # Add start tokens
            tokens = ['<START>', '<START>'] + tokens
            
            # Count trigrams
            for i in range(len(tokens) - 2):
                w1, w2, w3 = tokens[i], tokens[i+1], tokens[i+2]
                self.trigram_counts[w1][w2][w3] += 1
                self.bigram_counts[w1][w2] += 1
                self.vocab.add(w3)
    
    def probability(self, word, prev_word1, prev_word2):
        """Calculate P(word | prev_word1, prev_word2)"""
        bigram_count = self.bigram_counts[prev_word1][prev_word2]
        if bigram_count == 0:
            return 0
        return self.trigram_counts[prev_word1][prev_word2][word] / bigram_count
    
    def sentence_probability(self, tokens):
        """Calculate probability of a sentence"""
        tokens = ['<START>', '<START>'] + tokens
        prob = 1.0
        
        for i in range(2, len(tokens)):
            trigram_prob = self.probability(tokens[i], tokens[i-2], tokens[i-1])
            if trigram_prob == 0:
                return 0
            prob *= trigram_prob
        return prob
    
    def log_probability(self, tokens):
        """Calculate log probability"""
        tokens = ['<START>', '<START>'] + tokens
        log_prob = 0.0
        
        for i in range(2, len(tokens)):
            trigram_prob = self.probability(tokens[i], tokens[i-2], tokens[i-1])
            if trigram_prob == 0:
                return float('-inf')
            log_prob += np.log(trigram_prob)
        return log_prob

In [ ]:
# Train trigram model
trigram = TrigramModel()
trigram.train(train_tokens)

print("Sample trigram probabilities:")
print("\nP(word | 'natural', 'language'):")
if 'natural' in trigram.bigram_counts and 'language' in trigram.bigram_counts['natural']:
    for word, count in trigram.trigram_counts['natural']['language'].items():
        prob = trigram.probability(word, 'natural', 'language')
        print(f"  P('{word}' | 'natural', 'language') = {prob:.4f}")

In [ ]:
# Test trigram model
test_tokens = preprocess(test_sentences[0])
print(f"Test sentence: '{test_sentences[0]}'")
print(f"Tokens: {test_tokens}")
print(f"\nProbability: {trigram.sentence_probability(test_tokens):.2e}")
print(f"Log probability: {trigram.log_probability(test_tokens):.4f}")

## Problem: Zero Probabilities

If we encounter an unseen n-gram, the probability becomes 0, making the entire sentence probability 0.

**Solution**: Smoothing techniques

## 4. Smoothing Techniques

### Laplace Smoothing (Add-1 Smoothing)

Add 1 to all counts to ensure no zero probabilities.

**Formula**: P(wᵢ | wᵢ₋₁) = (Count(wᵢ₋₁, wᵢ) + 1) / (Count(wᵢ₋₁) + V)

where V is the vocabulary size.

In [ ]:
class BigramLaplaceSmoothing:
    def __init__(self):
        self.bigram_counts = defaultdict(Counter)
        self.unigram_counts = Counter()
        self.vocab = set()
    
    def train(self, tokenized_corpus):
        """Train bigram model with Laplace smoothing"""
        for tokens in tokenized_corpus:
            tokens = ['<START>'] + tokens
            
            for i in range(len(tokens) - 1):
                self.bigram_counts[tokens[i]][tokens[i+1]] += 1
                self.unigram_counts[tokens[i]] += 1
                self.vocab.add(tokens[i])
                self.vocab.add(tokens[i+1])
            
            self.unigram_counts[tokens[-1]] += 1
    
    def probability(self, word, previous_word):
        """Calculate P(word | previous_word) with Laplace smoothing"""
        V = len(self.vocab)
        numerator = self.bigram_counts[previous_word][word] + 1
        denominator = self.unigram_counts[previous_word] + V
        return numerator / denominator
    
    def sentence_probability(self, tokens):
        """Calculate probability of a sentence"""
        tokens = ['<START>'] + tokens
        prob = 1.0
        
        for i in range(1, len(tokens)):
            prob *= self.probability(tokens[i], tokens[i-1])
        return prob
    
    def log_probability(self, tokens):
        """Calculate log probability"""
        tokens = ['<START>'] + tokens
        log_prob = 0.0
        
        for i in range(1, len(tokens)):
            log_prob += np.log(self.probability(tokens[i], tokens[i-1]))
        return log_prob

In [ ]:
# Train bigram model with Laplace smoothing
bigram_laplace = BigramLaplaceSmoothing()
bigram_laplace.train(train_tokens)

# Test with unseen bigram
print("Comparison: Regular vs Laplace Smoothing")
print("\nFor seen bigram 'I' -> 'love':")
print(f"  Regular:  P('love' | 'I') = {bigram.probability('love', 'i'):.4f}")
print(f"  Laplace:  P('love' | 'I') = {bigram_laplace.probability('love', 'i'):.4f}")

print("\nFor unseen bigram 'I' -> 'hate':")
print(f"  Regular:  P('hate' | 'I') = {bigram.probability('hate', 'i'):.4f}")
print(f"  Laplace:  P('hate' | 'I') = {bigram_laplace.probability('hate', 'i'):.4f}")

### Add-k Smoothing

A generalization of Laplace smoothing where we add k (0 < k ≤ 1) instead of 1.

**Formula**: P(wᵢ | wᵢ₋₁) = (Count(wᵢ₋₁, wᵢ) + k) / (Count(wᵢ₋₁) + k×V)

Smaller k values preserve more of the original distribution.

In [ ]:
class BigramAddKSmoothing:
    def __init__(self, k=0.5):
        self.k = k
        self.bigram_counts = defaultdict(Counter)
        self.unigram_counts = Counter()
        self.vocab = set()
    
    def train(self, tokenized_corpus):
        """Train bigram model with Add-k smoothing"""
        for tokens in tokenized_corpus:
            tokens = ['<START>'] + tokens
            
            for i in range(len(tokens) - 1):
                self.bigram_counts[tokens[i]][tokens[i+1]] += 1
                self.unigram_counts[tokens[i]] += 1
                self.vocab.add(tokens[i])
                self.vocab.add(tokens[i+1])
            
            self.unigram_counts[tokens[-1]] += 1
    
    def probability(self, word, previous_word):
        """Calculate P(word | previous_word) with Add-k smoothing"""
        V = len(self.vocab)
        numerator = self.bigram_counts[previous_word][word] + self.k
        denominator = self.unigram_counts[previous_word] + self.k * V
        return numerator / denominator
    
    def sentence_probability(self, tokens):
        """Calculate probability of a sentence"""
        tokens = ['<START>'] + tokens
        prob = 1.0
        
        for i in range(1, len(tokens)):
            prob *= self.probability(tokens[i], tokens[i-1])
        return prob
    
    def log_probability(self, tokens):
        """Calculate log probability"""
        tokens = ['<START>'] + tokens
        log_prob = 0.0
        
        for i in range(1, len(tokens)):
            log_prob += np.log(self.probability(tokens[i], tokens[i-1]))
        return log_prob

In [ ]:
# Train models with different k values
k_values = [0.1, 0.5, 1.0]
models = {}

for k in k_values:
    model = BigramAddKSmoothing(k=k)
    model.train(train_tokens)
    models[k] = model

# Compare probabilities
print("Comparison of different k values (Add-k smoothing)")
print("\nFor seen bigram 'I' -> 'love':")
print(f"  No smoothing: {bigram.probability('love', 'i'):.4f}")
for k in k_values:
    prob = models[k].probability('love', 'i')
    print(f"  k={k}: {prob:.4f}")

print("\nFor unseen bigram 'I' -> 'hate':")
print(f"  No smoothing: {bigram.probability('hate', 'i'):.4f}")
for k in k_values:
    prob = models[k].probability('hate', 'i')
    print(f"  k={k}: {prob:.4f}")

## 5. Model Evaluation: Perplexity

Perplexity measures how well a language model predicts a test set.

**Lower perplexity = Better model**

**Formula**: Perplexity = 2^(-log₂(P(test_set)) / N)

where N is the number of words in the test set.

In [ ]:
def calculate_perplexity(model, test_sentences):
    """Calculate perplexity of a model on test sentences"""
    total_log_prob = 0
    total_words = 0
    
    for sentence in test_sentences:
        tokens = preprocess(sentence)
        log_prob = model.log_probability(tokens)
        
        if log_prob == float('-inf'):
            return float('inf')  # Infinite perplexity for zero probability
        
        total_log_prob += log_prob
        total_words += len(tokens)
    
    # Calculate perplexity
    avg_log_prob = total_log_prob / total_words
    perplexity = np.exp(-avg_log_prob)
    
    return perplexity

In [ ]:
# Calculate perplexity for different models
print("Perplexity Comparison on Test Set:")
print("="*50)

# Unigram
perp_uni = calculate_perplexity(unigram, test_sentences)
print(f"Unigram (no smoothing):        {perp_uni:.2f}")

# Bigram without smoothing
perp_bi = calculate_perplexity(bigram, test_sentences)
print(f"Bigram (no smoothing):         {perp_bi:.2f}")

# Bigram with Laplace smoothing
perp_bi_lap = calculate_perplexity(bigram_laplace, test_sentences)
print(f"Bigram (Laplace smoothing):    {perp_bi_lap:.2f}")

# Bigram with Add-k smoothing
for k in k_values:
    perp = calculate_perplexity(models[k], test_sentences)
    print(f"Bigram (Add-{k} smoothing):      {perp:.2f}")

# Trigram without smoothing
perp_tri = calculate_perplexity(trigram, test_sentences)
print(f"Trigram (no smoothing):        {perp_tri:.2f}")

## Visualization: Probability Distribution

In [ ]:
# Visualize probability distributions
word = 'i'
next_words = ['love', 'enjoy', 'hate']  # Mix of seen and unseen

# Get probabilities
prob_regular = [bigram.probability(w, word) for w in next_words]
prob_laplace = [bigram_laplace.probability(w, word) for w in next_words]
prob_add05 = [models[0.5].probability(w, word) for w in next_words]

x = np.arange(len(next_words))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width, prob_regular, width, label='No Smoothing', alpha=0.8)
ax.bar(x, prob_laplace, width, label='Laplace (k=1)', alpha=0.8)
ax.bar(x + width, prob_add05, width, label='Add-k (k=0.5)', alpha=0.8)

ax.set_xlabel('Next Word', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title(f'Probability Distribution: P(word | "{word}")', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(next_words)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Note: 'hate' is unseen in training data, so gets 0 probability without smoothing.")

## Summary Comparison Table

In [ ]:
# Create comparison table
test_sent = preprocess(test_sentences[0])

comparison_data = {
    'Model': ['Unigram', 'Bigram', 'Trigram', 'Bigram + Laplace', 'Bigram + Add-0.5'],
    'Probability': [
        f"{unigram.sentence_probability(test_sent):.2e}",
        f"{bigram.sentence_probability(test_sent):.2e}",
        f"{trigram.sentence_probability(test_sent):.2e}",
        f"{bigram_laplace.sentence_probability(test_sent):.2e}",
        f"{models[0.5].sentence_probability(test_sent):.2e}"
    ],
    'Perplexity': [
        f"{perp_uni:.2f}",
        f"{perp_bi:.2f}",
        f"{perp_tri:.2f}",
        f"{perp_bi_lap:.2f}",
        f"{calculate_perplexity(models[0.5], test_sentences):.2f}"
    ]
}

df_comparison = pd.DataFrame(comparison_data)
print(f"\nTest sentence: '{test_sentences[0]}'")
print("\nModel Comparison:")
print(df_comparison.to_string(index=False))

## Key Takeaways

### N-gram Models
- **Unigram**: Simple, assumes word independence
- **Bigram**: Considers one previous word (better context)
- **Trigram**: Considers two previous words (even better context, but data sparsity)

### Trade-offs
- **Higher n**: Better context but more sparse data, higher risk of zero probabilities
- **Lower n**: Less context but more robust to unseen sequences

### Smoothing
- **Purpose**: Handle unseen n-grams, prevent zero probabilities
- **Laplace (Add-1)**: Simple but can over-smooth
- **Add-k**: More flexible, smaller k preserves original distribution better

### Evaluation
- **Perplexity**: Lower is better
- Measures how "surprised" the model is by test data
- Good models assign high probabilities to test sentences

## Exercise for Students

1. Add more sentences to the training corpus and observe how perplexity changes
2. Implement 4-gram model
3. Experiment with different k values and find the optimal one
4. Try other smoothing techniques (Good-Turing, Kneser-Ney)
5. Generate text using the trained models (sampling from probability distributions)